# Retrieval of indexed documents based on DPR via OpenSearch

For indexes with **nested per-chunk embeddings** (`text_chunks_embedding`):
TREC Robust, NTCIR, SciDocs, and MS MARCO v1 **document**. (The flat MS MARCO
v1 **passage** index has its own notebook:
[opensearch_dpr_search_msmarco_passage.ipynb](opensearch_dpr_search_msmarco_passage.ipynb).)

#### Configuration

In [ ]:
index_name = "msmarco_v1_document_dpr"
model_id = "your-query-model-id"  # dense QUERY model: e5_en_query_model_id here; e5_ml_query_model_id for multilingual corpora
q = "do goldfish grow"
dataset_name = "msmarco-document/trec-dl-2019/judged"
# Also works unchanged for licensed corpora, e.g.:
#   index_name = "trec_robust_2005_dpr"; dataset_name = "aquaint/trec-robust-2005"

In [ ]:
import sys
!{sys.executable} -m pip install -q ir_datasets opensearch-py dotenv

In [ ]:
import pprint

Your opensearch password should be available in `~/.env`

```bash
    OPENSEARCH_INITIAL_ADMIN_PASSWORD="strong password"
```

In [ ]:
import os
from dotenv import load_dotenv
from opensearchpy import OpenSearch

load_dotenv()
host = 'localhost'
port = 9200
password = os.getenv("OPENSEARCH_INITIAL_ADMIN_PASSWORD")

client = OpenSearch(
    hosts=[{"host": host, "port": port}],
    http_auth=("admin", password),
    http_compress=True,
    use_ssl=True,
    verify_certs=False,
    ssl_assert_hostname=False,
    ssl_show_warn=False
)
pprint.pprint(client.info())

#### DPR Search

Documents were chunked and dense-encoded per passage into the nested
`text_chunks_embedding` field. A `neural` query encodes the query text
server-side with `model_id` and runs approximate k-NN against each passage
vector -- the `nested` wrapper with `score_mode: max` keeps the
best-matching chunk per document (MaxP).

In [ ]:
def build_query(query: str) -> dict:
    return {
        "nested": {
            "path": "text_chunks_embedding",
            "score_mode": "max",  # best-matching passage sets the doc score
            "query": {
                "neural": {
                    "text_chunks_embedding.knn": {
                        "query_text": query,
                        "model_id": model_id,
                        "k": 100
                    }
                }
            }
        }
    }

def search(query: str, size: int = 10) -> dict:
    return client.search(index=index_name, body={
        "size": size,
        "_source": ["docid", "title", "text"],
        "query": build_query(query),
    })

def show(resp, label=""):
    hits = resp["hits"]["hits"]
    print(f"\nTop {len(hits)} hits{' (' + label + ')' if label else ''}\n")
    for hit in hits:
        src = hit["_source"]
        print(f"[{src['docid']}] {src['title'][:50]}... (score={hit['_score']:.2f})")

In [ ]:
show(search(q, size=5), q)

#### Search with a Topic from the Dataset

In [ ]:
import importlib
import ir_datasets

# Register the local dataset module (e.g. "ntcir1-adhoc" -> ntcir1_adhoc.py)
dataset_dir = os.path.join(os.getcwd(), '..', 'dataset', dataset_name)
if os.path.isdir(dataset_dir):
    sys.path.append(dataset_dir)
    importlib.import_module(dataset_name.replace('-', '_'))

dataset = ir_datasets.load(dataset_name)

In [ ]:
topic = next(dataset.queries_iter())
pprint.pprint(topic)
# TREC-style topics carry .title; MS MARCO / TREC DL topics carry .text
topic_query = getattr(topic, "title", None) or topic.text
show(search(topic_query, size=5), f"topic {topic.query_id}: {topic_query}")

#### Rerank with the cross-encoder (common second stage)

Re-run the same first-stage query through the `rerank_bge_m3` search pipeline
(`BAAI/bge-reranker-v2-m3`, multilingual -- see
[ml_model_registration.ipynb](../indexing/opensearch/ml_model_registration.ipynb)).
Works identically over every index and ranker.

**Gotcha:** the request must return `_source` including the `text` field --
the rerank processor reads `document_fields: ["text"]` from `_source`; without
it every hit gets the same score and the order silently stays unchanged.

In [ ]:
def search_reranked(query: str, size: int = 10, rerank_pipeline: str = "rerank_bge_m3") -> dict:
    """Same first stage as search(), then cross-encoder reranking server-side."""
    return client.search(
        index=index_name,
        params={"search_pipeline": rerank_pipeline},
        body={
            "size": size,
            "_source": ["docid", "title", "text"],   # MUST include "text" (rerank context)
            "query": build_query(query),
            "ext": {"rerank": {"query_context": {"query_text": query}}},
        },
    )

show(search(q, size=5), "first stage")
show(search_reranked(q, size=5), "reranked")